In [ ]:
# =====================================
# Função de Aptidão
# =====================================

def contar_conflitos_pares(valores):
    """
    Conta todos os pares de valores iguais em um vetor.

    Exemplo:
    [5, 5, 5] possui 3 pares conflitantes:
    (5_1, 5_2), (5_1, 5_3) e (5_2, 5_3).
    """

    valores = valores[valores != 0]
    conflitos = 0

    for j in range(len(valores) - 1):
        for l in range(j + 1, len(valores)):
            if valores[j] == valores[l]:
                conflitos += 1

    return conflitos


def calcular_fitness(tabuleiro):
    """
    Calcula a quantidade total de conflitos
    nas linhas, colunas e blocos 3x3.

    Cada par de valores iguais é contabilizado
    como um conflito.

    Quanto menor o fitness, melhor a solução.
    O valor ótimo é fitness igual a zero.

    Cada chamada da função corresponde
    a uma avaliação da função objetivo.
    """
    global AVALIACOES

    # Conta uma avaliação da função objetivo
    AVALIACOES += 1

    tabuleiro = np.asarray(tabuleiro)

    if tabuleiro.shape != (9, 9):
        raise ValueError("O tabuleiro deve possuir dimensão 9x9.")

    conflitos = 0

    # ---------- Linhas ----------
    for linha in tabuleiro:
        conflitos += contar_conflitos_pares(linha)

    # ---------- Colunas ----------
    for coluna in tabuleiro.T:
        conflitos += contar_conflitos_pares(coluna)

    # ---------- Blocos 3x3 ----------
    for i in range(0, 9, 3):
        for j in range(0, 9, 3):
            bloco = tabuleiro[i:i + 3, j:j + 3].flatten()
            conflitos += contar_conflitos_pares(bloco)

    return conflitos


# =====================================
# Funções auxiliares
# =====================================

def copiar_tabuleiro(tabuleiro):
    """
    Retorna uma cópia independente do tabuleiro.
    """

    return copy.deepcopy(tabuleiro)


def vetor_para_tabuleiro(vetor, sudoku_original):
    """
    Converte um vetor de decisão em um tabuleiro de Sudoku.

    Cada valor do vetor é inserido em uma posição
    originalmente vazia do Sudoku.
    """

    if len(vetor) != len(posicoes_vazias):
        raise ValueError(
            "O tamanho do vetor deve ser igual ao número de posições vazias."
        )

    tabuleiro = sudoku_original.copy()

    for valor, (i, j) in zip(vetor, posicoes_vazias):
        tabuleiro[i, j] = valor

    return tabuleiro


def mostrar_tabuleiro(tabuleiro):
    """
    Exibe o tabuleiro de forma organizada.
    """

    for i in range(9):
        if i > 0 and i % 3 == 0:
            print("-" * 21)

        linha_formatada = []

        for j in range(9):
            if j > 0 and j % 3 == 0:
                linha_formatada.append("|")

            linha_formatada.append(str(tabuleiro[i, j]))

        print(" ".join(linha_formatada))



In [ ]:
# ============================================================
# CÉLULA 4 — ANT COLONY OPTIMIZATION (ACO)
# ============================================================


def contar_conflitos_locais_aco(tabuleiro, linha, coluna, valor):
    """
    Calcula quantos conflitos seriam gerados pela atribuição
    de determinado valor à célula (linha, coluna).

    São considerados conflitos:
    - na linha;
    - na coluna;
    - na subgrade 3x3.

    Essa contagem é utilizada apenas como informação
    heurística durante a construção da solução.
    """

    conflitos = 0

    # --------------------------------------------------------
    # Conflitos na linha
    # --------------------------------------------------------

    for j in range(9):

        # Desconsidera a própria célula avaliada
        if j == coluna:
            continue

        if tabuleiro[linha, j] == valor:
            conflitos += 1

    # --------------------------------------------------------
    # Conflitos na coluna
    # --------------------------------------------------------

    for i in range(9):

        # Desconsidera a própria célula avaliada
        if i == linha:
            continue

        if tabuleiro[i, coluna] == valor:
            conflitos += 1

    # --------------------------------------------------------
    # Conflitos na subgrade 3x3
    # --------------------------------------------------------

    inicio_linha = (linha // 3) * 3
    inicio_coluna = (coluna // 3) * 3

    for i in range(inicio_linha, inicio_linha + 3):
        for j in range(inicio_coluna, inicio_coluna + 3):

            # Desconsidera a própria célula
            if i == linha and j == coluna:
                continue

            if tabuleiro[i, j] == valor:
                conflitos += 1

    return conflitos


def calcular_heuristica_aco(tabuleiro, linha, coluna, valor):
    """
    Calcula a informação heurística eta associada à atribuição
    do valor v na célula (r, c).

    A heurística é definida por:

        eta = 1 / (1 + conflitos locais)

    Assim:
    - valores que não geram conflitos recebem eta = 1;
    - valores que geram conflitos recebem valores menores;
    - eta nunca é igual a zero.
    """

    conflitos = contar_conflitos_locais_aco(
        tabuleiro,
        linha,
        coluna,
        valor
    )

    eta = 1.0 / (1.0 + conflitos)

    return eta


def calcular_probabilidades_aco(
    tabuleiro,
    linha,
    coluna,
    feromonios,
    alpha,
    beta
):
    """
    Calcula a probabilidade de seleção dos valores de 1 a 9
    para a célula (linha, coluna).
    """

    # Valores candidatos para a célula
    candidatos = np.arange(1, 10)

    # Armazena a heurística de cada valor candidato
    heuristicas = np.zeros(9, dtype=float)

    for indice, valor in enumerate(candidatos):

        heuristicas[indice] = calcular_heuristica_aco(
            tabuleiro,
            linha,
            coluna,
            valor
        )

    # Feromônio associado a cada valor de 1 a 9
    trilhas_feromonio = feromonios[
        linha,
        coluna,
        1:10
    ]

    # Numerador da regra de transição
    pesos = (
        np.power(trilhas_feromonio, alpha)
        *
        np.power(heuristicas, beta)
    )

    soma_pesos = np.sum(pesos)

    # Distribuição uniforme de segurança
    if soma_pesos <= 0 or not np.isfinite(soma_pesos):

        probabilidades = np.full(
            shape=9,
            fill_value=1.0 / 9.0
        )

    else:

        probabilidades = pesos / soma_pesos

    return candidatos, probabilidades


def escolher_valor_aco(
    tabuleiro,
    linha,
    coluna,
    feromonios,
    alpha,
    beta
):
    """
    Seleciona probabilisticamente um valor de 1 a 9
    para a célula avaliada.
    """

    candidatos, probabilidades = calcular_probabilidades_aco(
        tabuleiro,
        linha,
        coluna,
        feromonios,
        alpha,
        beta
    )

    valor_escolhido = np.random.choice(
        candidatos,
        p=probabilidades
    )

    return int(valor_escolhido)


def construir_solucao_aco(
    sudoku_original,
    feromonios,
    alpha,
    beta
):
    """
    Constrói uma solução candidata completa.
    """

    tabuleiro = sudoku_original.copy()

    atribuicoes = []

    for linha, coluna in posicoes_vazias:

        valor = escolher_valor_aco(
            tabuleiro,
            linha,
            coluna,
            feromonios,
            alpha,
            beta
        )

        tabuleiro[linha, coluna] = valor

        atribuicoes.append(
            (linha, coluna, valor)
        )

    return tabuleiro, atribuicoes


def atualizar_feromonios_aco(
    feromonios,
    fitness_formigas,
    atribuicoes_formigas,
    rho
):
    """
    Atualiza as trilhas de feromônio conforme:

        tau(r,c,v) <- (1-rho) * tau(r,c,v)
                      + soma(delta / (1 + fitness))
    """

    # Evaporação
    feromonios *= (1.0 - rho)

    # Depósito de feromônio
    for fitness_formiga, atribuicoes in zip(
        fitness_formigas,
        atribuicoes_formigas
    ):

        deposito = 1.0 / (1.0 + fitness_formiga)

        for linha, coluna, valor in atribuicoes:

            feromonios[
                linha,
                coluna,
                valor
            ] += deposito

    return feromonios


def executar_aco(
    sudoku_original,
    quantidade_formigas=60,
    alpha=1.0,
    beta=2.0,
    rho=0.10,
    feromonio_inicial=1.0,
    exibir_progresso=True,
    intervalo_exibicao=100
):
    """
    Executa o algoritmo ACO para resolução do Sudoku.

    Critérios de parada:
    --------------------
    - fitness igual a zero; ou
    - número máximo de avaliações da função de aptidão.
    """

    # --------------------------------------------------------
    # Validação dos parâmetros
    # --------------------------------------------------------

    if quantidade_formigas <= 0:
        raise ValueError(
            "A quantidade de formigas deve ser positiva."
        )

    if alpha < 0 or beta < 0:
        raise ValueError(
            "Os parâmetros alpha e beta não podem ser negativos."
        )

    if not 0 < rho < 1:
        raise ValueError(
            "A taxa de evaporação rho deve estar entre 0 e 1."
        )

    if feromonio_inicial <= 0:
        raise ValueError(
            "O feromônio inicial deve ser maior que zero."
        )

    # --------------------------------------------------------
    # Início da medição do tempo
    # --------------------------------------------------------

    tempo_inicial = time.perf_counter()

    # --------------------------------------------------------
    # Inicialização da matriz de feromônios
    # --------------------------------------------------------

    feromonios = np.full(
        shape=(9, 9, 10),
        fill_value=feromonio_inicial,
        dtype=float
    )

    # Melhor solução encontrada durante toda a execução
    melhor_tabuleiro = None
    melhor_fitness = float("inf")

    # Histórico do melhor fitness global
    historico_fitness = []

    # Guarda a última iteração executada
    iteracao_final = 0

    # Contador de iterações
    iteracao = 0

    # --------------------------------------------------------
    # Laço principal do ACO
    # --------------------------------------------------------

    # Só inicia uma nova iteração se houver orçamento
    # suficiente para avaliar todas as formigas.
    while AVALIACOES + quantidade_formigas <= MAX_AVALIACOES:

        iteracao += 1
        iteracao_final = iteracao

        # Soluções construídas nesta iteração
        solucoes_formigas = []

        # Aptidão de cada solução
        fitness_formigas = []

        # Atribuições utilizadas por cada formiga
        atribuicoes_formigas = []

        # ----------------------------------------------------
        # Construção das soluções
        # ----------------------------------------------------

        for _ in range(quantidade_formigas):

            tabuleiro_formiga, atribuicoes = construir_solucao_aco(
                sudoku_original,
                feromonios,
                alpha,
                beta
            )

            # calcular_fitness incrementa AVALIACOES
            fitness_formiga = calcular_fitness(
                tabuleiro_formiga
            )

            solucoes_formigas.append(
                tabuleiro_formiga
            )

            fitness_formigas.append(
                fitness_formiga
            )

            atribuicoes_formigas.append(
                atribuicoes
            )

            # Atualiza o melhor resultado global
            if fitness_formiga < melhor_fitness:

                melhor_fitness = int(
                    fitness_formiga
                )

                melhor_tabuleiro = (
                    tabuleiro_formiga.copy()
                )

        # ----------------------------------------------------
        # Atualização dos feromônios
        # ----------------------------------------------------

        feromonios = atualizar_feromonios_aco(
            feromonios,
            fitness_formigas,
            atribuicoes_formigas,
            rho
        )

        # Registra o melhor fitness encontrado até o momento
        historico_fitness.append(
            melhor_fitness
        )

        # ----------------------------------------------------
        # Exibição do progresso
        # ----------------------------------------------------

        if exibir_progresso:

            deve_exibir = (
                iteracao == 1
                or iteracao % intervalo_exibicao == 0
                or melhor_fitness == 0
            )

            if deve_exibir:

                print(
                    f"Iteração {iteracao:>7} | "
                    f"Melhor fitness: {melhor_fitness} | "
                    f"Avaliações: {AVALIACOES}"
                )

        # ----------------------------------------------------
        # Critério de parada
        # ----------------------------------------------------

        if melhor_fitness == 0:
            break

    # --------------------------------------------------------
    # Finalização
    # --------------------------------------------------------

    tempo_execucao = (
        time.perf_counter() - tempo_inicial
    )

    resultado = {
        "algoritmo": "ACO",
        "tabuleiro": melhor_tabuleiro,
        "fitness": melhor_fitness,
        "iteracoes": iteracao_final,
        "avaliacoes": AVALIACOES,
        "tempo": tempo_execucao,
        "historico": historico_fitness,
        "solucionado": melhor_fitness == 0
    }

    return resultado

In [ ]:
# ============================================================
# CÉLULA 5 — ACO: 5 INSTÂNCIAS, 10 EXECUÇÕES POR INSTÂNCIA
# ============================================================

import numpy as np
import random
import copy
import time

# =====================================
# Instâncias
# =====================================

sudoku_dificil_1 = np.array([
    [6, 0, 0, 0, 8, 0, 0, 0, 0],
    [0, 0, 9, 7, 0, 0, 0, 2, 8],
    [2, 0, 0, 0, 0, 0, 3, 0, 0],

    [0, 0, 0, 0, 0, 0, 0, 8, 0],
    [0, 0, 0, 4, 5, 0, 0, 9, 0],
    [0, 9, 3, 0, 2, 0, 0, 0, 1],

    [0, 0, 0, 6, 0, 0, 2, 0, 0],
    [4, 0, 6, 0, 0, 9, 0, 0, 0],
    [0, 0, 2, 3, 0, 0, 0, 0, 4]
])

sudoku_dificil_2 = np.array([
    [0, 7, 0, 0, 1, 0, 0, 2, 0],
    [9, 0, 0, 7, 0, 5, 0, 0, 4],
    [0, 0, 1, 0, 0, 9, 6, 0, 0],
    [0, 1, 5, 0, 0, 0, 0, 6, 0],
    [4, 0, 0, 0, 5, 0, 0, 0, 1],
    [0, 6, 0, 0, 0, 0, 4, 7, 0],
    [0, 0, 4, 3, 0, 0, 7, 0, 0],
    [6, 0, 0, 1, 0, 8, 0, 0, 3],
    [0, 3, 0, 0, 9, 0, 0, 8, 0]
])

sudoku_dificil_3 = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0, 2],
    [0, 7, 0, 2, 0, 8, 5, 6, 0],
    [0, 0, 2, 0, 5, 0, 0, 0, 0],
    [0, 0, 0, 9, 0, 0, 0, 0, 0],
    [5, 0, 0, 6, 0, 2, 0, 0, 9],
    [7, 0, 1, 0, 0, 4, 6, 0, 3],
    [0, 1, 0, 0, 3, 0, 8, 0, 0],
    [0, 5, 0, 8, 0, 9, 0, 3, 0],
    [3, 0, 0, 0, 0, 0, 0, 0, 4]
])

sudoku_dificil_4 = np.array([
    [0, 0, 0, 0, 9, 6, 0, 0, 0],
    [7, 0, 4, 3, 0, 0, 0, 0, 5],
    [0, 0, 0, 5, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 6, 0, 0],
    [5, 0, 0, 2, 0, 0, 4, 0, 9],
    [0, 9, 0, 0, 8, 0, 0, 7, 0],
    [0, 4, 2, 9, 7, 0, 0, 0, 0],
    [0, 0, 8, 4, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 8, 0, 0, 0]
])

sudoku_dificil_5 = np.array([
    [0, 0, 2, 0, 0, 0, 8, 0, 0],
    [0, 8, 0, 6, 1, 0, 0, 9, 0],
    [7, 0, 0, 0, 0, 0, 0, 0, 6],
    [0, 4, 0, 5, 0, 8, 0, 3, 9],
    [0, 0, 0, 0, 2, 0, 0, 0, 0],
    [8, 5, 0, 4, 0, 3, 0, 2, 0],
    [6, 0, 0, 0, 0, 0, 0, 0, 2],
    [0, 9, 0, 0, 4, 1, 0, 6, 0],
    [0, 0, 7, 0, 0, 0, 4, 0, 0]
])

instancias_faceis = {
    "sudoku_dificil_1": sudoku_dificil_1,
    "sudoku_dificil_2": sudoku_dificil_2,
    "sudoku_dificil_3": sudoku_dificil_3,
    "sudoku_dificil_4": sudoku_dificil_4,
    "sudoku_dificil_5": sudoku_dificil_5
}

# =====================================
# Controle de avaliações
# =====================================

NUM_EXECUCOES = 10
MAX_AVALIACOES = 400000

resultados_aco = []

PARAMETROS_ACO = {
    "quantidade_formigas": 60,
    "alpha": 0.1,
    "beta": 8.0,
    "rho": 0.8,
    "feromonio_inicial": 1.0,
    "exibir_progresso": False,
    "intervalo_exibicao": 100
}

for nome_instancia, sudoku_instancia in instancias_faceis.items():

    print(f"\n===== {nome_instancia} =====")

    # Atualiza as posições para a instância atual
    posicoes_fixas = list(zip(*np.where(sudoku_instancia != 0)))
    posicoes_vazias = list(zip(*np.where(sudoku_instancia == 0)))


    for execucao in range(NUM_EXECUCOES):

        # Usa a mesma sequência de sementes em cada instância
        SEED_EXECUCAO = execucao

        random.seed(SEED_EXECUCAO)
        np.random.seed(SEED_EXECUCAO)

        # Reinicia o contador antes de cada execução
        AVALIACOES = 0

        resultado = executar_aco(
            sudoku_original=sudoku_instancia.copy(),
            **PARAMETROS_ACO
        )

        # Armazena a instância e os resultados da execução
        resultados_aco.append({
            "instancia": nome_instancia,
            "execucao": execucao + 1,
            "seed": SEED_EXECUCAO,
            "fitness": resultado["fitness"],
            "iteracoes": resultado["iteracoes"],
            "avaliacoes": resultado["avaliacoes"],
            "tempo": resultado["tempo"],
            "solucionado": resultado["solucionado"]
        })

        print(
            f"Instância: {nome_instancia} | "
            f"Execução {execucao + 1:02d} | "
            f"Seed: {SEED_EXECUCAO:02d} | "
            f"Fitness: {resultado['fitness']} | "
            f"Iterações: {resultado['iteracoes']} | "
            f"Avaliações: {resultado['avaliacoes']} | "
            f"Tempo: {resultado['tempo']:.2f}s | "
            f"Solucionado: {resultado['solucionado']}"
        )


===== sudoku_dificil_4 =====
Instância: sudoku_dificil_4 | Execução 01 | Seed: 00 | Fitness: 2 | Iterações: 6666 | Avaliações: 399960 | Tempo: 3616.50s | Solucionado: False
Instância: sudoku_dificil_4 | Execução 02 | Seed: 01 | Fitness: 4 | Iterações: 6666 | Avaliações: 399960 | Tempo: 3596.16s | Solucionado: False
Instância: sudoku_dificil_4 | Execução 03 | Seed: 02 | Fitness: 4 | Iterações: 6666 | Avaliações: 399960 | Tempo: 3613.45s | Solucionado: False
Instância: sudoku_dificil_4 | Execução 04 | Seed: 03 | Fitness: 2 | Iterações: 6666 | Avaliações: 399960 | Tempo: 3621.14s | Solucionado: False
Instância: sudoku_dificil_4 | Execução 05 | Seed: 04 | Fitness: 2 | Iterações: 6666 | Avaliações: 399960 | Tempo: 3684.02s | Solucionado: False
Instância: sudoku_dificil_4 | Execução 06 | Seed: 05 | Fitness: 2 | Iterações: 6666 | Avaliações: 399960 | Tempo: 3703.66s | Solucionado: False


In [ ]:
# ============================================================
# RESUMO DAS 30 EXECUÇÕES DO ACO
# ============================================================

fitness = [r["fitness"] for r in resultados_aco]
tempos = [r["tempo"] for r in resultados_aco]
avaliacoes = [r["avaliacoes"] for r in resultados_aco]

sucessos = sum(r["solucionado"] for r in resultados_aco)

taxa_sucesso = (sucessos / len(resultados_aco)) * 100

print("\n" + "=" * 50)
print("RESUMO — ACO")
print("=" * 50)

print(f"Execuções: {len(resultados_aco)}")
print(f"Sucessos: {sucessos}")
print(f"Taxa de sucesso: {taxa_sucesso:.2f}%")

print(f"\nMelhor fitness: {min(fitness)}")
print(f"Pior fitness: {max(fitness)}")
print(f"Fitness médio: {np.mean(fitness):.2f}")
print(f"Mediana do fitness: {np.median(fitness):.2f}")
print(f"Desvio padrão do fitness: {np.std(fitness):.2f}")

print(f"\nTempo médio: {np.mean(tempos):.2f} segundos")
print(f"Mediana do tempo: {np.median(tempos):.2f} segundos")

print(f"\nMédia de avaliações: {np.mean(avaliacoes):.2f}")